<a href="https://colab.research.google.com/github/P-franczak/colab-deep-hough-transform/blob/main/Hanqer_Deep_Hough_Transform_Train_SEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorboardX
!pip install POT
!pip install Ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.7 MB/s eta 0:00:00


In [2]:
import os
from google.colab import drive

# Define a base path in Google Drive for persistent storage
# This will create a 'colab_persistent_data' folder in your Google Drive.
drive_base_path = '/content/drive/MyDrive/colab_persistent_data'

# Mount Google Drive
if not os.path.exists('/content/drive'):
  print("Mounting Google Drive...")
  drive.mount('/content/drive')
else:
  print("Google Drive is already mounted.")

# Ensure the base directory in Drive exists
if not os.path.exists(drive_base_path):
  print(f"Creating base directory '{drive_base_path}' in Google Drive for persistent storage.")
  !mkdir -p "$drive_base_path"

Mounting Google Drive...
Mounted at /content/drive


In [3]:
# Define paths for the repository
repo_name = 'deep-hough-transform'
repo_drive_path = os.path.join(drive_base_path, repo_name)
repo_local_path = repo_name # The directory name after cloning

if os.path.exists(repo_drive_path):
  print(f"Repository '{repo_name}' found in Google Drive. Copying to local environment...")
  !cp -r "$repo_drive_path" .
else:
  print(f"Repository '{repo_name}' not found in Google Drive. Cloning and saving to Drive for future sessions...")
  if not os.path.exists(repo_local_path):
    !git clone https://github.com/P-franczak/deep-hough-transform.git
  # Copy the cloned repository to Google Drive
  !cp -r "$repo_local_path" "$drive_base_path"
  print(f"Repository '{repo_name}' cloned and saved to '{drive_base_path}'.")

Repository 'deep-hough-transform' not found in Google Drive. Cloning and saving to Drive for future sessions...
Cloning into 'deep-hough-transform'...
remote: Enumerating objects: 650, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 650 (delta 64), reused 46 (delta 40), pack-reused 547 (from 1)
Receiving objects: 100% (650/650), 13.88 MiB | 17.05 MiB/s, done.
Resolving deltas: 100% (133/133), done.
Repository 'deep-hough-transform' cloned and saved to '/content/drive/MyDrive/colab_persistent_data'.


In [4]:
# Define paths for the NKL dataset zip file
zip_filename = 'ICCV2017_JTLEE_dataset.7z'
zip_drive_path = os.path.join(drive_base_path, zip_filename)
zip_local_path = zip_filename # The filename in the current directory after wget

%cd deep-hough-transform/

if os.path.exists(zip_drive_path):
  print(f"'{zip_filename}' found in Google Drive. Copying to local environment...")
  # Copy to the current directory, which is /content/deep-hough-transform/ after 'cd'

  !cp "$zip_drive_path" .
else:
  print(f"'{zip_filename}' not found in Google Drive. Downloading and saving to Drive for future sessions...")
  !wget https://mcl.korea.ac.kr/research/Submitted/jtlee_slnet/ICCV2017_JTLEE_dataset.7z
  # Copy the downloaded zip file to Google Drive
  !cp "$zip_local_path" "$drive_base_path"
  print(f"'{zip_filename}' downloaded and saved to '{drive_base_path}'.")

/content/deep-hough-transform
'ICCV2017_JTLEE_dataset.7z' not found in Google Drive. Downloading and saving to Drive for future sessions...
--2026-05-01 09:14:09--  https://mcl.korea.ac.kr/research/Submitted/jtlee_slnet/ICCV2017_JTLEE_dataset.7z
Resolving mcl.korea.ac.kr (mcl.korea.ac.kr)... 163.152.175.73
Connecting to mcl.korea.ac.kr (mcl.korea.ac.kr)|163.152.175.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 183635100 (175M) [application/x-7z-compressed]
Saving to: ‘ICCV2017_JTLEE_dataset.7z’

ICCV2017_JTLEE_data 100%[===================>] 175.13M  20.1MB/s    in 9.2s    

2026-05-01 09:14:19 (19.1 MB/s) - ‘ICCV2017_JTLEE_dataset.7z’ saved [183635100/183635100]

'ICCV2017_JTLEE_dataset.7z' downloaded and saved to '/content/drive/MyDrive/colab_persistent_data'.


In [5]:
!7z x ICCV2017_JTLEE_dataset.7z


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan         1 file, 183635100 bytes (176 MiB)

Extracting archive: ICCV2017_JTLEE_dataset.7z
--
Path = ICCV2017_JTLEE_dataset.7z
Type = 7z
Physical Size = 183635100
Headers Size = 30835
Method = LZMA2:25
Solid = +
Blocks = 1

  0%      3% 35 - ICCV2017_JTLEE_images/0036.jpg                                          7% 98 - ICCV2017_JTLEE_images/0099.jpg                                         10% 149 - ICCV2017_JTLEE_images/0150.jpg                                          14% 196 - ICCV2017

In [6]:
%cd model/_cdht

/content/deep-hough-transform/model/_cdht


In [8]:
!python setup.py build

running build
running build_ext
W0501 09:17:15.027000 3383 torch/utils/cpp_extension.py:535] There are no x86_64-linux-gnu-g++ version bounds defined for CUDA version 12.8
building 'deep_hough' extension
ninja: no work to do.
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 /content/deep-hough-transform/model/_cdht/build/temp.linux-x86_64-cpython-312/deep_hough_cuda.o /content/deep-hough-transform/model/_cdht/build/temp.linux-x86_64-cpython-312/deep_hough_cuda_kernel.o -L/usr/local/lib/python3.12/dist-packages/torch/lib -L/usr/local/cuda/lib64 -L/usr/lib/x86_64-linux-gnu -lc10 -ltorch -ltorch_cpu -ltorch_python -lcudart -lc10_cuda -ltorch_cuda -o build/lib.linux-x86_64-cpython-312/deep_hough.cpython-312-x86_64-linux-gnu.so


In [9]:
!pip install -e . --user

Obtaining file:///content/deep-hough-transform/model/_cdht
  Preparing metadata (setup.py) ... done
  Running setup.py develop for deep_hough


In [10]:
%cd ../..

/content/deep-hough-transform


In [11]:
!python data/prepare_data_JTLEE.py --root './ICCV2017_JTLEE_images/' --label './ICCV2017_JTLEE_gtlines_all' --save-dir './data/training/JTLEE_resize_100_100/' --list './data/training/JTLEE.lst' --prefix 'JTLEE_resize_100_100' --fixsize 400 --numangle 100 --numrho 100

Processing 1132 [1/1716]...
[ WARN:0@7.799] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
Processing 1492 [2/1716]...
Processing 0085 [3/1716]...
Processing 0892 [4/1716]...
Processing 1509 [5/1716]...
Processing 0793 [6/1716]...
Processing 0237 [7/1716]...
Processing 0132 [8/1716]...
Processing 0391 [9/1716]...
Processing 1172 [10/1716]...
Processing 1517 [11/1716]...
Processing 1526 [12/1716]...
Processing 1252 [13/1716]...
Processing 0738 [14/1716]...
Processing 0965 [15/1716]...
Processing 0621 [16/1716]...
Processing 1150 [17/1716]...
Processing 1278 [18/1716]...
Processing 1375 [19/1716]...
Processing 1347 [20/1716]...
Processing 0348 [21/1716]...
Processing 0515 [22/1716]...
Processing 0888 [23/1716]...
Processing 0269 [24/1716]...
Processing 0414 [25/1716]...
Processing 0577 [26/1716]...
Processing 0257 [27/1716]...
Processing 1518 [28/1716]...
Processing 1668 [29/1716]...
Processing 1244 [30/1716]...
Processing 0977 [31/

In [16]:
!python train.py --resume result/reproduce/checkpoint.pth.tar

2026-05-01 10:21:45,548 INFO {'DATA': {'DIR': 'data/training/', 'VAL_DIR': 'data/training/', 'TEST_DIR': 'data', 'LABEL_FILE': 'data/training/train_1716_100_100.txt', 'VAL_LABEL_FILE': 'data/training/test_1716_100_100.txt', 'TEST_LABEL_FILE': 'data/sel_test.txt', 'BATCH_SIZE': 8, 'WORKERS': 2}, 'OPTIMIZER': {'LR': 0.0002, 'MOMENTUM': 0.9, 'GAMMA': 0.1, 'WEIGHT_DECAY': 0.0, 'STEPS': []}, 'MODEL': {'NUMANGLE': 100, 'NUMRHO': 100, 'FIX': True, 'THRESHOLD': 0.01, 'EDGE_ALIGN': False, 'BACKBONE': 'resnet50'}, 'TRAIN': {'EPOCHS': 30, 'PRINT_FREQ': 100, 'TEST': False, 'SEED': 1997, 'GPU_ID': 0, 'DATA_PARALLEL': False, 'RESUME': None}, 'MISC': {'TMP': './result/reproduce'}}
2026-05-01 10:21:45,548 INFO Namespace(config='./config.yml', resume='result/reproduce/checkpoint.pth.tar', tmp='')
2026-05-01 10:21:46,341 INFO => loading checkpoint 'result/reproduce/checkpoint.pth.tar'
Traceback (most recent call last):
  File "/content/deep-hough-transform/train.py", line 326, in <module>
    main()
  F